In [ ]:
import os
import json
import random
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Iterator
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
from tqdm.auto import tqdm
import torch
import torch.nn as nn

In [ ]:
def load_cloudsen_npz(path: str) -> np.ndarray:
    with np.load(path) as npz:
        if "x" in npz.files:
            return npz["x"]

In [ ]:
class CloudSENReconDataset(Dataset):
    def __init__(self, base_dir: str, ids: List[int],
                 target_idxs: List[int],
                 target_hw: Tuple[int, int] = (512, 512),
                 to_float: bool = True):
        self.base_dir = base_dir
        self.to_float = to_float
        self.target_hw = target_hw
        self.target_idxs = sorted(int(i) for i in target_idxs)
        files = []
        for i in ids:
            fname = f"{int(i):05d}.npz"
            path = os.path.join(base_dir, fname)
            if os.path.isfile(path):
                files.append((i, path))
        self.files = files

    def __len__(self) -> int:
        return len(self.files)


    def __getitem__(self, index: int):
      img_id, path = self.files[index]
      arr = load_cloudsen_npz(path)
      X = torch.from_numpy(arr).float()
      C, H, W = X.shape
      keep = torch.ones(C, dtype=torch.bool)
      keep[self.target_idxs] = False

      Xin = X[keep, :, :]
      y = X[self.target_idxs, :, :]

      meta = {"image_id": int(img_id), "path": path}
      return Xin, y, meta


In [ ]:
class HeadDW7x7(nn.Module):
    def __init__(self, c_in: int, hidden: int = 32, out_ch: int = 1):
        super().__init__()
        self.pw1 = nn.Conv2d(c_in, hidden, 1, bias=True)
        self.dw = nn.Conv2d(hidden, hidden, 7, padding=3, groups=hidden, abias=True)
        self.pw2 = nn.Conv2d(hidden, out_ch, 1, bias=True)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.act(self.pw1(x))
        x = self.act(self.dw(x))
        x = self.pw2(x)
        return x


In [ ]:
class CUDAPrefetcher:
    def __init__(self, loader, device):
        self.base_loader = loader
        self.device = device
        self.stream = torch.cuda.Stream()
        self.reset()

    def reset(self):
        self.loader = iter(self.base_loader)
        self.next_batch = None
        self._preload()

    def _preload(self):
        try:
            Xin, y, meta = next(self.loader)
        except StopIteration:
            self.next_batch = None
            return
        with torch.cuda.stream(self.stream):
            Xin = Xin.to(
                self.device,
                dtype=torch.float32,
                memory_format=torch.channels_last,
                non_blocking=True,
            )
            y = y.to(
                self.device,
                dtype=torch.float32,
                memory_format=torch.channels_last,
                non_blocking=True,
            )
            self.next_batch = (Xin, y, meta)

    def __iter__(self):
        self.reset()
        return self

    def __next__(self):
        torch.cuda.current_stream().wait_stream(self.stream)
        if self.next_batch is None:
            raise StopIteration
        batch = self.next_batch
        self._preload()
        return batch

In [ ]:
def build_gpu_tensor_dataset(ds: Dataset, device: torch.device):
    Xs, Ys, metas = [], [], []
    for i in range(len(ds)):
        Xin, y, meta = ds[i]
        Xs.append(Xin.unsqueeze(0))
        Ys.append(y.unsqueeze(0))
        metas.append(meta)
    X = torch.cat(Xs, dim=0).to(
        device=device,
        dtype=torch.float32,
        memory_format=torch.channels_last,
        non_blocking=True,
    )
    Y = torch.cat(Ys, dim=0).to(
        device=device,
        dtype=torch.float32,
        memory_format=torch.channels_last,
        non_blocking=True,
    )
    return torch.utils.data.TensorDataset(X, Y), metas

In [ ]:
def _get_xy_from_batch(batch, preload_gpu: bool):
    if preload_gpu:
        Xin, y = batch
    else:
        Xin, y, _ = batch
    return Xin, y

In [ ]:
def build_model(c_in: int, arch: str, hidden: int, out_ch: int) -> nn.Module:
    a = arch.lower()

    if a in ("dw7x7", "7x7"):
        return HeadDW7x7(c_in, hidden, out_ch)

    else:
        raise ValueError(f"Unknown arch: {arch}")


In [ ]:
def make_loader(ds: Dataset, batch_size=8, shuffle=False, num_workers=4, pin_memory=True):
    kw = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )
    if num_workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(ds, **kw)

In [ ]:
def list_ids_from_folder(data_dir: str) -> List[int]:
    ids = []
    for fname in os.listdir(data_dir):
        if not fname.endswith(".npz"):
            continue
        stem = os.path.splitext(fname)[0]
        try:
            ids.append(int(stem))
        except ValueError:
            continue
    ids = sorted(ids)
    return ids

In [ ]:
@dataclass
class TrainConfig:
    train_dir: str
    val_dir: str
    target_idxs: List[int]
    target_hw: Tuple[int, int] = (512, 512)
    arch: str = "1x1"
    hidden: int = 32
    batch_size: int = 8
    epochs: int = 50
    lr: float = 1e-3
    weight_decay: float = 0.0
    num_workers: int = 2
    seed: int = 1337
    save_prefix: str = "./adapter_7to6"
    amp: bool = True
    prefetch_gpu: bool = True
    preload_gpu: bool = False


In [ ]:
def set_seed(seed: int = 1337):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
@torch.no_grad()
def compute_r2_rmse(y_true: torch.Tensor, y_pred: torch.Tensor):
    yt = y_true.float().reshape(-1).cpu().numpy()
    yp = y_pred.float().reshape(-1).cpu().numpy()
    if r2_score is None or mean_squared_error is None:
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)
        r2 = 1.0 - (ss_res / (ss_tot + 1e-12))
        rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
        return float(r2), rmse
    r2 = float(r2_score(yt, yp))
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    return r2, rmse



@torch.no_grad()
def compute_r2_rmse_per_channel(y_true: torch.Tensor, y_pred: torch.Tensor):
    assert y_true.shape == y_pred.shape
    _, C, _, _ = y_true.shape
    r2s, rmses = [], []
    for c in range(C):
        r2, rmse = compute_r2_rmse(y_true[:, c], y_pred[:, c])
        r2s.append(r2)
        rmses.append(rmse)
    mean_r2 = float(np.mean(r2s))
    mean_rmse = float(np.mean(rmses))
    return mean_r2, mean_rmse, r2s, rmses

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import json
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm


def train_adapter(cfg: TrainConfig) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

    assert cfg.target_idxs and len(cfg.target_idxs) > 0

    train_ids = list_ids_from_folder(cfg.train_dir)
    val_ids = list_ids_from_folder(cfg.val_dir) if cfg.val_dir is not None else []

    ds_tr = CloudSENReconDataset(cfg.train_dir, train_ids, cfg.target_idxs, cfg.target_hw)
    ds_val = CloudSENReconDataset(cfg.val_dir, val_ids, cfg.target_idxs, cfg.target_hw) if val_ids else None

    c_in = ds_tr[0][0].shape[0]
    out_ch = ds_tr[0][1].shape[0]

    model = build_model(c_in, cfg.arch, cfg.hidden, out_ch)
    model = model.to(device).to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))
    loss_fn = nn.SmoothL1Loss()

    if cfg.preload_gpu:
        ds_tr_gpu, _ = build_gpu_tensor_dataset(ds_tr, device)
        loader_tr = DataLoader(
            ds_tr_gpu,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False,
        )

        loader_val = None
        if ds_val is not None:
            ds_val_gpu, _ = build_gpu_tensor_dataset(ds_val, device)
            loader_val = DataLoader(
                ds_val_gpu,
                batch_size=cfg.batch_size,
                shuffle=False,
                num_workers=0,
                pin_memory=False,
            )

        use_prefetch = False
    else:
        loader_tr = make_loader(
            ds_tr,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
        )
        loader_val = (
            make_loader(
                ds_val,
                batch_size=cfg.batch_size,
                shuffle=False,
                num_workers=cfg.num_workers,
                pin_memory=(device.type == "cuda"),
            )
            if ds_val is not None
            else None
        )
        use_prefetch = (device.type == "cuda" and cfg.prefetch_gpu)

    best_val_r2 = -1e9
    best_state = None
    best_val_metrics = None
    last_val_metrics = None

    for ep in range(cfg.epochs):
        model.train()
        total_loss = 0.0
        n_pix = 0

        data_iter = CUDAPrefetcher(loader_tr, device) if use_prefetch else loader_tr
        data_iter = tqdm(data_iter, desc=f"Epoch {ep+1}/{cfg.epochs} [train]", leave=False)

        for batch in data_iter:
            Xin, y = _get_xy_from_batch(batch, cfg.preload_gpu)

            Xin = Xin / 10000.0
            y = y / 10000.0

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(cfg.amp and device.type == "cuda")):
                pred = model(Xin)
                loss = loss_fn(pred, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            bs, _, h, w = Xin.shape
            total_loss += float(loss.detach().cpu()) * bs * h * w
            n_pix += bs * h * w

        avg_loss = total_loss / max(1, n_pix)

        val_metrics = None
        if loader_val is not None:
            model.eval()
            val_iter = CUDAPrefetcher(loader_val, device) if use_prefetch else loader_val
            val_iter = tqdm(val_iter, desc=f"Epoch {ep+1}/{cfg.epochs} [val]", leave=False)

            y_true_all, y_pred_all = [], []
            with torch.no_grad(), torch.cuda.amp.autocast(enabled=(cfg.amp and device.type == "cuda")):
                for batch in val_iter:
                    Xin, y = _get_xy_from_batch(batch, cfg.preload_gpu)
                    Xin = Xin / 10000.0
                    y = y / 10000.0
                    p = model(Xin)
                    y_true_all.append(y.detach().cpu())
                    y_pred_all.append(p.detach().cpu())

            if y_true_all:
                y_true = torch.cat(y_true_all, dim=0)
                y_pred = torch.cat(y_pred_all, dim=0)

                r2_mean, rmse_mean, r2s, rmses = compute_r2_rmse_per_channel(y_true, y_pred)

                val_metrics = {
                    "val_R2_macro": float(r2_mean),
                    "val_RMSE_macro": float(rmse_mean),
                    "per_band_R2": {str(b): float(r2s[i]) for i, b in enumerate(cfg.target_idxs)},
                    "per_band_RMSE": {str(b): float(rmses[i]) for i, b in enumerate(cfg.target_idxs)},
                }

                last_val_metrics = val_metrics

                if r2_mean > best_val_r2:
                    best_val_r2 = float(r2_mean)
                    best_val_metrics = val_metrics
                    best_state = {
                        "state_dict": model.state_dict(),
                        "arch": cfg.arch,
                        "hidden": cfg.hidden,
                        "target_idxs": list(map(int, cfg.target_idxs)),
                        "target_hw": cfg.target_hw,
                    }

        if val_metrics is not None:
            print(
                f"Epoch {ep+1}/{cfg.epochs} | "
                f"train_L1pp={avg_loss:.6f} | "
                f"val_R2_macro={val_metrics['val_R2_macro']:.4f}"
            )
        else:
            print(f"Epoch {ep+1}/{cfg.epochs} | train_L1pp={avg_loss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state["state_dict"])

    os.makedirs(os.path.dirname(cfg.save_prefix), exist_ok=True)

    torch.save(
        {
            "state_dict": model.state_dict(),
            "arch": cfg.arch,
            "hidden": cfg.hidden,
            "target_idxs": list(map(int, cfg.target_idxs)),
            "target_hw": cfg.target_hw,
        },
        f"{cfg.save_prefix}_model.pt",
    )

    if best_val_metrics is not None:
        with open(f"{cfg.save_prefix}_metrics.json", "w") as f:
            json.dump(best_val_metrics, f, indent=2)

    return best_val_metrics if best_val_metrics is not None else {"info": "Training finished"}


In [ ]:
cfg = TrainConfig(
    train_dir="/content/drive/MyDrive/data/cloudsen12_data/cloudsen12_npz_13bands/train",
    val_dir="/content/drive/MyDrive/data/cloudsen12_data/cloudsen12_npz_13bands/val",
    target_idxs=[],
    arch="dw7x7",
    hidden=16,
    batch_size=32,
    epochs=50,
    lr=2e-3,
    weight_decay=0.0,
    num_workers=5,
    seed=1337,
    save_prefix="",
    amp=True,
    prefetch_gpu=True,
    preload_gpu=False,
)
